# 01 — OAT (One-At-A-Time ablation)

Reference 1점에서 한 축씩만 갈아끼우며 marginal RMSE 측정. 11축 × 옵션 = reference 1 + 변형 31 = **32 cell × 1 seed = 32 fit** (seed는 LGBM-default가 결정론적이라 1개로 충분 — axes.SEEDS).

- agg_preset 축(12개)은 reg_level='position' reference에서 no-op이라 그 셀들만 reg_level='unit'으로 흔든다.
- impute=knn 셀은 fit당 ~70분 (다른 셀의 ~20배) — group study에선 제외, 여기서만 1셀 측정.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*.csv`
- **출력**: `4_output/baseline/oat/{master.csv, checkpoint.json}`
- **참조**: [strategy.md §4](strategy.md), [strategy_common.md §6·§7·§8](../strategy_common.md)
- **Resume**: master.csv에 이미 있는 (axis, option, seed) 조합은 자동 skip

## 1. 환경 설정 + 데이터 로드

In [1]:
import os, sys

# Colab이면 코드/데이터 zip을 Drive에서 받아 풀고 PROJECT_ROOT를 잡음, 로컬이면 ../../setup.py만
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system('gdown 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system('gdown 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
    PROJECT_ROOT = '/content/project'
except ImportError:
    %run ../../setup.py
    from utils.config import PROJECT_ROOT

# 0_baseline 폴더를 경로에 추가 → `import axes` 가 이 노트북 옆의 axes.py를 찾게 (다른 단계와 격리)
BASELINE_DIR = os.path.join(PROJECT_ROOT, '3_modeling', '0_baseline')
if BASELINE_DIR not in sys.path:
    sys.path.insert(0, BASELINE_DIR)

import warnings
warnings.filterwarnings('ignore')

from utils.data import load_all, get_feat_cols, split_xs
import axes   # OAT 한 셀(cfg) 실행기 — axes.AXES(11축 옵션) / axes.run_one / axes.generate_oat_grid

xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
axes.set_data(xs, xs_dict, ys, feat_cols)   # 모든 run_one 호출이 같은 데이터를 쓰도록 모듈에 한 번 주입

print(f'Feature 수: {len(feat_cols)}')
print(f'Die 수: train={len(xs_dict["train"]):,}, val={len(xs_dict["validation"]):,}, test={len(xs_dict["test"]):,}')

setup 완료
[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
Feature 수: 1087
Die 수: train=104,748, val=34,908, test=34,916


## 2. 출력 경로 + master.csv / checkpoint 로드

- `master.csv` row 단위 영구 박제 (axis/option/seed/oof_rmse/val_rmse/test_rmse/elapsed_sec/timestamp)
- `checkpoint.json` — 완료한 (axis, option, seed) 튜플 set
- 끊김 후 재실행 시 master.csv에 있는 조합은 자동 skip

In [2]:
# 노트북 상단 단일 파라미터 (모델 학습 병렬도 + 트리 개수 고정)
N_JOBS = 5
N_ESTIMATORS = 100  # LGBM n_estimators 고정 — OAT는 HP가 아니라 전처리 축 효과만 본다

import json
import pandas as pd
from datetime import datetime

OUT_DIR = os.path.join(PROJECT_ROOT, '4_output', 'baseline', 'oat')
os.makedirs(OUT_DIR, exist_ok=True)
MASTER_CSV = os.path.join(OUT_DIR, 'master.csv')      # 결과 1행씩 누적 박제 (끊겨도 이어서)
CKPT_JSON  = os.path.join(OUT_DIR, 'checkpoint.json') # 마지막 완료 정보 (진행 상황 확인용)
META_JSON  = os.path.join(OUT_DIR, 'meta.json')       # 재현성 메타 (run마다 덮어씀)

# master.csv 컬럼: 결과 메트릭 + cfg 11축을 'cfg_<axis>'로 풀어 저장 → 행 하나로 셀 cfg 완전 복원
CFG_COLS = [f'cfg_{ax}' for ax in axes.AXES.keys()]
MASTER_COLS = [
    'axis', 'option', 'seed', 'is_reference',
    'oof_rmse', 'val_rmse', 'test_rmse',
    'elapsed_sec', 'effective_target_transform', 'timestamp',
] + CFG_COLS

# 기존 master.csv가 있으면 로드해서 "이미 끝낸 (axis, option, seed)" 집합을 만든다 → 재실행 시 건너뜀
if os.path.exists(MASTER_CSV):
    master = pd.read_csv(MASTER_CSV)
    done_set = set(zip(master['axis'], master['option'].astype(str), master['seed']))
    print(f'기존 master.csv: {len(master)} rows, done={len(done_set)} (axis,option,seed)')
else:
    master = pd.DataFrame()
    done_set = set()
    print('master.csv 신규 생성')

# meta.json — 이번 run의 설정 전체를 한 파일에 박제 (reference cfg, 11축 옵션, seed들, 고정 전처리 등)
meta = {
    'created':       datetime.now().isoformat(timespec='seconds'),
    'n_jobs':        N_JOBS,
    'n_estimators':  N_ESTIMATORS,
    'reference':     axes.REFERENCE,
    'axes':          {k: list(map(str, v)) for k, v in axes.AXES.items()},
    'seeds':         axes.SEEDS,
    'agg_preset_lib': axes.AGG_PRESET_LIB,
    'tweedie_losses': list(axes.TWEEDIE_LOSSES),  # 이 loss들이면 target_transform 자동 OFF
    'pp_pin': {
        'cleaning': axes.PP_PIN_CLEANING,
        'outlier':  axes.PP_PIN_OUTLIER,
        'binarize': axes.PP_PIN_BINARIZE,
        'iso':      axes.PP_PIN_ISO,
        'lds':      axes.PP_PIN_LDS,
        'ge':       axes.PP_PIN_GE,
    },
    'exclude_cols':  axes.EXCLUDE_COLS,
    'master_cols':   MASTER_COLS,
}
with open(META_JSON, 'w') as f:
    json.dump(meta, f, indent=2, default=str, ensure_ascii=False)
print(f'meta.json saved → {META_JSON}')

master.csv 신규 생성
meta.json saved → c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\baseline\oat\meta.json


## 3. OAT grid 순회 — 32 cell (seed 1개)

axes.generate_oat_grid()로 (axis, option, seed, cfg, is_reference) 튜플 생성. 매 fit 완료 시 row append + master.csv 즉시 flush (끊김 보호).

In [3]:
# OAT grid 전체 = (reference 1점 + 각 축의 각 옵션) × seed들. 이미 끝낸 조합은 빼고 todo만 남김
grid = axes.generate_oat_grid()
todo = [(axis, opt, seed, cfg, is_ref) for (axis, opt, seed, cfg, is_ref) in grid
        if (axis, str(opt), seed) not in done_set]

print(f'전체 grid: {len(grid)} | 완료: {len(grid) - len(todo)} | 남은 cell-seed: {len(todo)}')

전체 grid: 32 | 완료: 0 | 남은 cell-seed: 32


In [4]:
for i, (axis_name, option, seed, cfg, is_ref) in enumerate(todo):
    print(f'\n[{i+1}/{len(todo)}] axis={axis_name} option={option} seed={seed}')
    try:
        # 이 cfg(=reference에서 한 축만 갈아끼운 설정)로 전처리+LGBM 5-fold 학습 → oof/val/test RMSE
        result = axes.run_one(
            cfg, seed=seed, n_jobs=N_JOBS, n_estimators=N_ESTIMATORS,
        )
    except Exception as e:
        # 에러를 삼키지 말고 즉시 보고 (어느 조합에서 깨졌는지 알 수 있게)
        print(f'!! ERROR axis={axis_name} option={option} seed={seed}: {e}')
        raise

    row = {
        'axis': axis_name,
        'option': option,
        'seed': seed,
        'is_reference': is_ref,
        'oof_rmse':  result['oof_rmse'],
        'val_rmse':  result['val_rmse'],
        'test_rmse': result['test_rmse'],
        'elapsed_sec': result['elapsed_sec'],
        'effective_target_transform': result['effective_target_transform'],  # tweedie loss면 'none'으로 override됐는지 추적
        'timestamp': datetime.now().isoformat(timespec='seconds'),
    }
    # cfg 11축을 cfg_CLF, cfg_reg_level, ... 로 풀어 같이 저장 (행 하나로 셀 완전 복원)
    for ax_name in axes.AXES.keys():
        row[f'cfg_{ax_name}'] = cfg[ax_name]

    master = pd.concat([master, pd.DataFrame([row])], ignore_index=True)
    master[MASTER_COLS].to_csv(MASTER_CSV, index=False)   # 매 cell마다 즉시 flush — 중간에 끊겨도 여기까지는 보존

    done_set.add((axis_name, str(option), seed))
    with open(CKPT_JSON, 'w') as f:
        json.dump({
            'done_count': len(done_set),
            'total': len(grid),
            'last_completed': row,
        }, f, indent=2, default=str)

    print(f'  oof={result["oof_rmse"]:.6f}  val={result["val_rmse"]:.6f}  '
          f'test={result["test_rmse"]:.6f}  elapsed={result["elapsed_sec"]:.1f}s'
          f'  effective_tt={result["effective_target_transform"]}')

print(f'\n[OAT 완료] master.csv: {len(master)} rows')


[1/32] axis=reference option=off seed=42
Rerun: using best_pp_params_resolved (from trial.user_attrs) — conditional skip 키도 보존됨
Rerun preprocessing: cleaning=12 args, outlier method=winsorize, binarize_apply=False, iso_enabled=False, lds_enabled=False, ge_use_encoder=False, agg_funcs=['mean', 'std', 'range', 'min', 'max', 'median']
클리닝 파이프라인 시작
원본 feature 수: 1087
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 982개
    컬럼: 1087 → 982 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=40%
  제거: 5개, 잔여: 977개
    컬럼: 982 → 977 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 950개
    컬럼: 977 → 950 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 347개, 잔여: 603개
    컬럼: 950 → 603 (347개 제거)
    DataFrame: (104748, 607)

[결측 indicator] 9개 컬럼 추가 (결측률 >= 5%)
[공간 보간 imputation] 총 결측: 343,494
  train-only 모드: train 104,748 / 전체 174,572 행
  1단계 (공간 보간, dist<=5.0): 156,772개 채움 → 잔여: 186,722
  2단계 (lot 평균, train 기준): 105,526개 채